# 01 — Monthly Data Audit\n\nThis notebook audits the CRSP CIZ Monthly Stock File production extract **before** any signal or performance analysis.\n\nThe raw WRDS file remains local and is excluded from GitHub by `.gitignore`.

In [ ]:
from pathlib import Path\nimport sys\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()\nsys.path.append(str(ROOT))\n\nfrom src.clean_monthly import CleaningConfig, build_clean_monthly_panel\nfrom src.concentration import calculate_monthly_concentration, largest_stocks_by_month

## 1. Load raw production extract

In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'\nraw = pd.read_csv(RAW, low_memory=False)\nraw['MthCalDt'] = pd.to_datetime(raw['MthCalDt'])\n\nprint(f'Rows: {len(raw):,}')\nprint(f'Unique PERMNO: {raw.PERMNO.nunique():,}')\nprint(f'Months: {raw.MthCalDt.nunique():,}')\nprint(f'Date range: {raw.MthCalDt.min().date()} to {raw.MthCalDt.max().date()}')\nraw.head()

## 2. Check the WRDS query filter

In [ ]:
for col in ['USIncFlg','IssuerType','SecurityType','SecuritySubType','ShareType','ShrAdrFlg','PrimaryExch']:\n    print('\\n', col)\n    print(raw[col].value_counts(dropna=False).head(20))

## 3. Duplicate-key audit

In [ ]:
key = ['PERMNO', 'MthCalDt']\ndup = raw[raw.duplicated(key, keep=False)].sort_values(key)\nprint(f'Rows in duplicated keys: {len(dup):,}')\nprint(f'Duplicated key groups: {dup.groupby(key).ngroups:,}')\nprint(f'Exact full-row duplicates: {raw.duplicated().sum():,}')

## 4. Missingness audit

In [ ]:
missing = (\n    raw.isna().sum()\n    .to_frame('missing_n')\n    .assign(missing_pct=lambda x: 100 * x.missing_n / len(raw))\n    .sort_values('missing_n', ascending=False)\n)\nmissing.head(20)

## 5. Build the pre-specified clean monthly panel

In [ ]:
config = CleaningConfig(min_price=5.0, nyse_size_percentile=0.20)\nclean, audit = build_clean_monthly_panel(RAW, config)\naudit

## 6. Number of securities through time

In [ ]:
n_by_month = clean.groupby('MthCalDt')['PERMNO'].nunique()\nfig, ax = plt.subplots(figsize=(10, 4))\nn_by_month.plot(ax=ax)\nax.set_title('Eligible securities after primary sample screens')\nax.set_xlabel('Month')\nax.set_ylabel('Number of securities')\nplt.show()

## 7. Calculate concentration measures

In [ ]:
conc = calculate_monthly_concentration(clean)\nconc.tail()

In [ ]:
for column, title in [('Top10Share', 'Top-10 market-cap share'), ('HHI', 'Market-cap HHI')]:\n    fig, ax = plt.subplots(figsize=(10, 4))\n    ax.plot(conc['MthCalDt'], conc[column])\n    ax.set_title(title)\n    ax.set_xlabel('Month')\n    ax.set_ylabel(column)\n    plt.show()

## 8. Diagnostics only\n\nInspect the largest stocks in selected months to verify that the concentration series is economically sensible. This is **not** a hypothesis test.

In [ ]:
largest_stocks_by_month(\n    clean,\n    n=10,\n    months=['2000-01-31', '2010-01-29', '2020-01-31', '2025-12-31'],\n)